In [1]:
# 设置数据显示格式
import pandas as pd
pd.set_option('display.expand_frame_repr',False)
pd.set_option('display.max_columns',None)


In [2]:
# 读取并查看数据
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(df.head())
print(df.info())


   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService     MultipleLines InternetService OnlineSecurity OnlineBackup DeviceProtection TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling              PaymentMethod  MonthlyCharges TotalCharges Churn
0  7590-VHVEG  Female              0     Yes         No       1           No  No phone service             DSL             No          Yes               No          No          No              No  Month-to-month              Yes           Electronic check           29.85        29.85    No
1  5575-GNVDE    Male              0      No         No      34          Yes                No             DSL            Yes           No              Yes          No          No              No        One year               No               Mailed check           56.95       1889.5    No
2  3668-QPYBK    Male              0      No         No       2          Yes                No             DSL            Yes  

In [3]:
# 1. 挑选核心特征和目标
# 注意：我们先避开 gender, Partner 等文字列，只拿数字列
features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Contract']
target = 'Churn'

# 2. 创建简化版的 X 和 y
X = df[features].copy()
y = df[target].copy()
# 3. 处理一个隐藏的“大坑”：TotalCharges 的数据类型
# 原始数据里这一列有时会有空格，导致它被识别为 object，我们要强制转成数字
X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')

# 4. 把目标 y 转成 0 和 1 (逻辑回归只认数字)
y = y.apply(lambda x: 1 if x == 'Yes' else 0)

X = pd.get_dummies(X, columns=['Contract'], drop_first=True)

print("数据已简化！当前的特征列：", X.columns.tolist())
print("现在的 X 预览：\n", X.head())
print(X.dtypes)

数据已简化！当前的特征列： ['tenure', 'MonthlyCharges', 'TotalCharges', 'Contract_One year', 'Contract_Two year']
现在的 X 预览：
    tenure  MonthlyCharges  TotalCharges  Contract_One year  Contract_Two year
0       1           29.85         29.85              False              False
1      34           56.95       1889.50               True              False
2       2           53.85        108.15              False              False
3      45           42.30       1840.75               True              False
4       2           70.70        151.65              False              False
tenure                 int64
MonthlyCharges       float64
TotalCharges         float64
Contract_One year       bool
Contract_Two year       bool
dtype: object


In [4]:
# 导入工具
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# 划分数据集
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# 定义Pipeline步骤
steps = [
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler()),
    ('logreg',LogisticRegression())
]
pipeline = Pipeline(steps)

# 设置网格搜索参数
params = {'logreg__C':[0.01, 0.1, 1, 10, 100]}

# 开启网格搜索
cv = GridSearchCV(pipeline,param_grid=params,cv=5)

# 训练模型
cv.fit(X_train,y_train)

# 打印结果
print(f"最佳参数:{cv.best_params_}")
print(f"训练集最高得分:{cv.best_score_:.4f}")
print(f"测试集准确率:{cv.score(X_test,y_test):.4f}")

最佳参数:{'logreg__C': 0.01}
训练集最高得分:0.7897
测试集准确率:0.8041
